In [ ]:
import numpy as np,pandas as pd ,seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

first of all load our data

In [ ]:
df = pd.read_csv('Data.csv')

## Basic INFO

In [ ]:
print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().sum())
print(df['Churn'].value_counts())
print (df.shape)
print(df.columns)

In [ ]:
df = df.drop(columns=['gender','customerID'])
df.head()

now we will seperate the numeric and categorical columns

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(np.mean(df['TotalCharges']))
numeric_columns = ['MonthlyCharges','tenure','TotalCharges']

now we speperated the numeric but before going to seperate the categorical cols we first clean the up 

In [ ]:
un_cleaned_catagorical = []
for col in df.columns:
    if df[col].dtype == object or df[col].dtype == 'string':
        if df[col].nunique() > 2:
            un_cleaned_catagorical.append(col)
to_clean = ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
for col in to_clean:
    df[col] = df[col].map({'No internet service': 'No','No':'No','Yes':'Yes','No phone service':'No'})


In [ ]:
df['Churn'] = df['Churn'].map({'Yes':1,'No':0})
x = df.drop('Churn',axis=1)
y= df['Churn']

In [ ]:
x['SeniorCitizen'] = x['SeniorCitizen'].map({0:'No',1:'Yes'})
to_Label_encode = []
to_onhot_encode = []
to_scale = numeric_columns
for col in x.columns:
    if x[col].nunique() == 2:
        to_Label_encode.append(col)
    elif x[col].nunique() == 3 or x[col].nunique() == 4:
        to_onhot_encode.append(col)
print(to_Label_encode,to_onhot_encode)

no we do our furthure splits and make pipeline

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=42)

In [ ]:
Preprocessor =  ColumnTransformer([
    ('scaler', StandardScaler(), to_scale),
    ('hot_encoder', OneHotEncoder(handle_unknown='ignore'), to_onhot_encode),
    ('binary_encoder', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), to_Label_encode)
])
models = {
    'LogisticRegression':LogisticRegression(class_weight='balanced',max_iter=6000),
    'KNN':KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced')
}
param_grids = {
    'LogisticRegression': {
        'solver':  ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag'],
        'C': [ 0.1,0.01,1,30],
        'penalty': ['l1', 'l2']
    },
    'KNN': {
        'n_neighbors':[21,23,29],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    'NaiveBayes': {
        'var_smoothing': [1e-9, 1e-8, 1e-7] 
    },
    'DecisionTree': {
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split':[2,5,10],
        'min_samples_leaf': [1, 2, 4]
    }
}

grid search cv 

In [ ]:
best_models = {}
for model_name in models:
    print(f'filhal ==> {model_name } <== trian ho rha he ...........')
    my_pipeline = Pipeline([
        ('preprocessor',Preprocessor),
        ('model',models[model_name])
    ])
    current_grid = {}
    for prams, val in param_grids[model_name].items():
        current_grid[f'model__{prams}'] = val
    search = GridSearchCV(
        estimator=my_pipeline,
        param_grid=current_grid,
        cv=5,
        scoring='f1',
        n_jobs=-1,
    )
    search.fit(x_train, y_train)
    best_models[model_name] = search.best_estimator_
    print(f"Optimal Parameters for {model_name}: {search.best_params_}")
    print(f"Highest CV Score: {search.best_score_:.4f}\n")

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score,confusion_matrix

print("======= FINAL TEST SCORES =======")

for model_name, trained_model in best_models.items():

    prediction = trained_model.predict(x_test)

    print(f"\n{model_name}")
    print("-" * 30)
    print("confusion_matrix :", confusion_matrix(y_test, prediction))
    print("Accuracy :", accuracy_score(y_test, prediction))
    print("Precision:", precision_score(y_test, prediction))
    print("Recall   :", recall_score(y_test, prediction))
    print("F1 Score :", f1_score(y_test, prediction))

now we are saving our model through our joblib


In [ ]:
import joblib

final_model = best_models['LogisticRegression']

joblib.dump(final_model,'trained_model.pkl')